# Partie 3 — Spark SQL et DataFrames

Projet Écosystème Big Data 2 — FastFood.ma — EMSI 5ème année, 2026-2027.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").exists():
    root = Path("/work")
sys.path[:0] = [str(root / "src"), str(root / "third_party")]


Le schéma est un `StructType` écrit à la main (`fastfood/schemas.py`). Pas d'inférence : une colonne manquante doit faire échouer le job, pas devenir une chaîne.

Le référentiel restaurants tient en 200 lignes. On le broadcast. Le plan physique contient `BroadcastHashJoin` et `BroadcastExchange`. Le fichier complet est dans `outputs/partie3_explain.txt`.


In [ ]:
from fastfood.partie3 import run
from fastfood.session import build_spark

spark = build_spark("fastfood-partie3-sql", delta=True)
result = run(spark)
result["seconds"], result["partitions"]


## UDF contre `when`

`distance_band` en UDF Python traverse la frontière JVM / interpréteur à chaque ligne. Catalyst ne la voit pas, Tungsten ne la compile pas.
La même règle en `when / otherwise` reste dans le plan. Sur l'historique, le temps de l'UDF est plusieurs fois celui de l'expression native. Les deux produisent les mêmes bandes : court &lt; 2 km, moyen de 2 à 5 km, long au-delà.

## RDD, DataFrame, Dataset

| | RDD | DataFrame | Dataset |
| --- | --- | --- | --- |
| Type | objets Python | colonnes + schéma | objets JVM typés |
| Optimiseur | aucun | Catalyst + Tungsten | Catalyst + Tungsten |
| Ici | partie 2 | parties 3 à 6 | pas d'API Python |

Sur 50 000 lignes déjà en cache, le RDD chaud et le DataFrame chaud sont du même ordre de grandeur (environ 0,2 s). On ne choisit pas le DataFrame pour ce chrono. On le choisit parce que le schéma, la jointure broadcast, Catalyst et le streaming sont le même moteur. L'écart mesuré est ailleurs : `when/otherwise` est environ deux fois plus rapide que l'UDF Python sur la tranche de distance.
